# Day 7 | Hands-On 3: Z-ORDER vs Liquid Clustering on Fact Tables
### GlobalMart Data Engineering Bootcamp

| | |
|---|---|
| **Follows** | ILT 2 — Performance in Modelling |
| **Source** | `gbmart.gold.fact_sales` (read-only) |
| **Target** | Your own practice copies — never the shared `fact_sales` |
| **Duration** | 60 minutes |

### Learning Objectives
- Apply Z-ORDER to a fact-table-shaped table using columns chosen from real query patterns (ILT 2), not guesswork
- Measure the before/after on an actual join-heavy business query, not a simple filter
- Compare Z-ORDER against Liquid Clustering on the same data

---
**Before You Start — Cost & Safety Note**

> You saw the mechanics of `OPTIMIZE`/`ZORDER`/Liquid Clustering on a generic practice table back in Day 4 HOL 2. Today applies the exact same commands specifically to a `fact_sales`-shaped table — but **still only ever against your own personal copy**, never the shared `gbmart.gold.fact_sales` table itself. Replace `YOUR_SCHEMA` below with something that's actually yours (e.g. `main.virinchy_practice`) before running anything. Nothing in this HOL starts a cluster, job, or pipeline — everything runs as ordinary cells on your already-running cluster.

---
## Phase 1 — Copy `fact_sales` Into Your Own Practice Schema

In [ ]:
# ─── SETUP ─────────────────────────────────────────────────────────────────
# Replace YOUR_SCHEMA with a schema you own -- never point this at gbmart.*
import time
from pyspark.sql.functions import col

PRACTICE_SCHEMA = "main.YOUR_SCHEMA"   # <- e.g. "main.virinchy_practice"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PRACTICE_SCHEMA}")

FACT_TABLE   = f"{PRACTICE_SCHEMA}.fact_sales_practice"
LIQUID_TABLE = f"{PRACTICE_SCHEMA}.fact_sales_liquid_practice"
print(f"Practice targets: {FACT_TABLE}, {LIQUID_TABLE}")

In [ ]:
# ─── Read real fact_sales (read-only, safe) and deliberately fragment it ────
# repartition(200) simulates the small-file state fact_sales would reach
# after months of incremental appends (Day 9-10's MERGE-based refresh),
# without waiting months to actually get there.

source_df = spark.table("gbmart.gold.fact_sales")
print(f"Source rows: {source_df.count():,}")

(
    source_df
    .repartition(200)
    .write.format("delta").mode("overwrite")
    .saveAsTable(FACT_TABLE)
)

file_count = spark.sql(f"DESCRIBE DETAIL {FACT_TABLE}").select("numFiles").collect()[0][0]
print(f"Practice fact table written with {file_count} files (deliberately fragmented).")

---
## Phase 2 — Baseline: Time the Real Business Query, Not a Simple Filter

Day 4 HOL 2 measured a single-column filter. Today measures something closer to what actually happens in production: **the exact join-and-aggregate shape of `vw_monthly_category_sales`** from HOL 2, run against the fragmented practice table.

In [ ]:
def run_category_query(fact_table_name):
    """Same shape as vw_monthly_category_sales, pointed at a practice table."""
    return spark.sql(f"""
        SELECT p.category, d.month, SUM(f.Sales_amount) AS revenue
        FROM {fact_table_name} f
        JOIN gbmart.gold.dim_product p ON f.Product_ID = p.product_id AND p.is_current = true
        JOIN gbmart.gold.dim_date d    ON f.Time_ID = d.date_key
        GROUP BY p.category, d.month
    """)

start = time.time()
result_count = run_category_query(FACT_TABLE).count()
baseline_seconds = time.time() - start

print(f"Result rows : {result_count}")
print(f"Baseline query time : {baseline_seconds:.3f}s")
print(f"File count before OPTIMIZE : {file_count}")

---
## Phase 3 — OPTIMIZE + Z-ORDER on the Columns ILT 2 Identified

ILT 2 identified `Product_ID` and `Time_ID` as the right Z-ORDER columns for exactly this query shape — not a guess, a conclusion drawn from the real view definitions.

In [ ]:
spark.sql(f"OPTIMIZE {FACT_TABLE} ZORDER BY (Product_ID, Time_ID)")

file_count_after = spark.sql(f"DESCRIBE DETAIL {FACT_TABLE}").select("numFiles").collect()[0][0]
print(f"File count after OPTIMIZE + ZORDER : {file_count_after}  (was {file_count})")

start = time.time()
result_count = run_category_query(FACT_TABLE).count()
zorder_seconds = time.time() - start

print(f"Result rows : {result_count}")
print(f"Query time after OPTIMIZE+ZORDER : {zorder_seconds:.3f}s  (baseline was {baseline_seconds:.3f}s)")
print("On a practice-scale table the gap may be modest -- it grows sharply at GlobalMart's real fact_sales scale.")

---
## Phase 4 — Liquid Clustering as an Alternative

Same idea as Day 4 HOL 2, applied here to the fact-table shape: a **new** table with `CLUSTER BY`, so future `OPTIMIZE` calls stay cheap incrementally as new order lines arrive — relevant here specifically because Day 9–10 turns `fact_sales` into an incrementally-refreshed table.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {LIQUID_TABLE}")
spark.sql(f"""
    CREATE TABLE {LIQUID_TABLE}
    CLUSTER BY (Product_ID, Time_ID)
    AS SELECT * FROM {FACT_TABLE}
""")

# OPTIMIZE on a clustered table applies clustering incrementally -- no ZORDER BY needed
spark.sql(f"OPTIMIZE {LIQUID_TABLE}")

start = time.time()
result_count = run_category_query(LIQUID_TABLE).count()
liquid_seconds = time.time() - start

print(f"Result rows : {result_count}")
print(f"Liquid Clustering query time : {liquid_seconds:.3f}s")
print()
print(f"{'Baseline (fragmented)':<28} {baseline_seconds:.3f}s")
print(f"{'OPTIMIZE + ZORDER':<28} {zorder_seconds:.3f}s")
print(f"{'Liquid Clustering':<28} {liquid_seconds:.3f}s")

---
## Phase 5 — Why This Matters More Here Than in Day 4

Day 4's practice table was a one-time static build. `fact_sales` is not — Day 9–10 refreshes it incrementally via `MERGE` every time new orders land, meaning **new small files get added continuously**, exactly the fragmentation pattern this HOL just simulated. This is precisely why Liquid Clustering is the more natural fit for `fact_sales` specifically: it re-clusters incrementally on every `OPTIMIZE`, without needing to re-specify `ZORDER BY` columns or worry about full-table rewrites growing more expensive as the table grows.

---
## Cleanup (Optional)

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {FACT_TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {LIQUID_TABLE}")
# print("Practice tables dropped -- the real gbmart.gold.fact_sales was never touched.")

---
## Key Takeaways

1. **Z-ORDER columns should come from real query patterns** (ILT 2's view definitions), not guesswork.
2. **Measuring a join-and-aggregate query is more realistic than a simple filter** for a fact table — that's how `fact_sales` actually gets queried in production.
3. **Liquid Clustering is the better long-term fit for `fact_sales`** specifically because it will be incrementally refreshed (Day 9–10) — it stays cheap to re-cluster as new data arrives.
4. **Every experiment ran against a personal practice copy** — running `OPTIMIZE`/`ZORDER` against the real shared `gbmart.gold.fact_sales` mid-cohort would affect everyone else's work.

### Submission Checklist
- [ ] Practice copies built in your own schema, not `gbmart.*`
- [ ] Baseline, Z-ORDER, and Liquid Clustering timings all captured
- [ ] Can explain why Liquid Clustering fits `fact_sales`'s future (incremental refresh) better than one-time Z-ORDER